# Parte Comum: EDA do Obesity (estrutural, distribucional e outliers)

Dataset: Obesity (UCI 544), arquivo `data/parte_comum/obesity.csv`.

Pilares cobertos neste notebook: **estrutural**, **distribucional** e **outliers**.
O pilar relacional fica em `eda_relacional_comum.ipynb`.

In [ ]:
# [IA] Claude Code – importacao das bibliotecas, leitura do dataset e pasta de saida
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

CAMINHO_DADOS = Path("../../data/parte_comum/obesity.csv")
PASTA_SAIDA = Path("../../out/parte_comum")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CAMINHO_DADOS)
COLUNA_CLASSE = "NObeyesdad"
df.head()

In [ ]:
# [IA] Claude Code – ordem das classes (do menor para o maior nivel de peso), usada em todos os graficos
ORDEM_CLASSES = [
    "Insufficient_Weight", "Normal_Weight",
    "Overweight_Level_I", "Overweight_Level_II",
    "Obesity_Type_I", "Obesity_Type_II", "Obesity_Type_III",
]
assert set(df[COLUNA_CLASSE].unique()) == set(ORDEM_CLASSES)

## 1. Pilar estrutural

### 1.1 Dimensões

In [ ]:
# [IA] Claude Code – quantidade de linhas e colunas
linhas, colunas = df.shape
print(f"Linhas: {linhas}")
print(f"Colunas: {colunas}")

### 1.2 Tipos das variáveis

In [ ]:
# [IA] Claude Code – tabela de tipos: dtype do pandas, quantidade de valores distintos e exemplos
tipos = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "qtd_valores_distintos": df.nunique(),
    "exemplos": [df[c].drop_duplicates().head(4).tolist() for c in df.columns],
})
tipos["grupo_pandas"] = ["numerica" if pd.api.types.is_numeric_dtype(df[c]) else "categorica" for c in df.columns]
tipos

In [ ]:
# [IA] Claude Code – significado de cada coluna (dicionario da UCI) e classificacao estatistica
dicionario = pd.DataFrame([
    ("Gender", "Genero", "qualitativa nominal"),
    ("Age", "Idade (anos)", "quantitativa continua"),
    ("Height", "Altura (m)", "quantitativa continua"),
    ("Weight", "Peso (kg)", "quantitativa continua"),
    ("family_history_with_overweight", "Familiar com sobrepeso", "qualitativa nominal (binaria)"),
    ("FAVC", "Consome alimentos caloricos com frequencia", "qualitativa nominal (binaria)"),
    ("FCVC", "Frequencia de consumo de vegetais (1 a 3)", "quantitativa discreta / escala ordinal"),
    ("NCP", "Numero de refeicoes principais por dia (1 a 4)", "quantitativa discreta / escala ordinal"),
    ("CAEC", "Come entre as refeicoes", "qualitativa ordinal"),
    ("SMOKE", "Fuma", "qualitativa nominal (binaria)"),
    ("CH2O", "Consumo diario de agua (1 a 3)", "quantitativa discreta / escala ordinal"),
    ("SCC", "Monitora as calorias", "qualitativa nominal (binaria)"),
    ("FAF", "Frequencia de atividade fisica (0 a 3)", "quantitativa discreta / escala ordinal"),
    ("TUE", "Tempo usando aparelhos eletronicos (0 a 2)", "quantitativa discreta / escala ordinal"),
    ("CALC", "Consumo de alcool", "qualitativa ordinal"),
    ("MTRANS", "Meio de transporte", "qualitativa nominal"),
    ("NObeyesdad", "Nivel de obesidade (variavel de classe)", "qualitativa ordinal (alvo)"),
], columns=["coluna", "significado", "classificacao"]).set_index("coluna")
dicionario

### Classificação das variáveis

_(texto do grupo)_

### 1.3 Qualidade dos dados

In [ ]:
# [IA] Claude Code – valores ausentes (NaN) por coluna
ausentes = df.isna().sum()
print("Total de valores ausentes (NaN):", ausentes.sum())
ausentes[ausentes > 0]

In [ ]:
# [IA] Claude Code – linhas duplicadas (todas as colunas iguais)
duplicadas = df[df.duplicated(keep=False)]
print("Linhas duplicadas (copias extras):", df.duplicated().sum())
print("Linhas envolvidas em duplicacao (original + copias):", len(duplicadas))
duplicadas.sort_values(list(df.columns)).head(10)

In [ ]:
# [IA] Claude Code – categorias de cada variavel categorica (procura de grafias inconsistentes)
colunas_categoricas = df.select_dtypes(exclude="number").columns
for coluna in colunas_categoricas:
    print(f"{coluna}: {df[coluna].value_counts().to_dict()}")

In [ ]:
# [IA] Claude Code – escalas que deveriam ser inteiras (questionario) mas tem valores fracionados
# A UCI informa que 77% dos dados foram gerados sinteticamente (SMOTE) e 23% vieram de questionario.
# O SMOTE interpola entre pontos, por isso aparecem valores como FCVC = 2.45.
ESCALAS = {"FCVC": (1, 3), "NCP": (1, 4), "CH2O": (1, 3), "FAF": (0, 3), "TUE": (0, 2)}

fracionados = pd.DataFrame({
    "faixa_esperada": [f"{a} a {b}" for a, b in ESCALAS.values()],
    "min": [df[c].min() for c in ESCALAS],
    "max": [df[c].max() for c in ESCALAS],
    "qtd_nao_inteiros": [(df[c] % 1 != 0).sum() for c in ESCALAS],
}, index=list(ESCALAS))
fracionados["pct_nao_inteiros"] = (fracionados["qtd_nao_inteiros"] / len(df) * 100).round(1)
fracionados

In [ ]:
# [IA] Claude Code – idade tambem aparece fracionada pelo mesmo motivo
print("Idades nao inteiras:", (df["Age"] % 1 != 0).sum(), f"({(df['Age'] % 1 != 0).mean() * 100:.1f}%)")

In [ ]:
# [IA] Claude Code – faixas das variaveis numericas (procura de valores impossiveis)
colunas_numericas = df.select_dtypes(include="number").columns.tolist()
faixas = df[colunas_numericas].describe().T[["min", "max", "mean", "std"]].round(3)
faixas["qtd_negativos"] = (df[colunas_numericas] < 0).sum()
faixas

### 1.4 Variável de classe

In [ ]:
# [IA] Claude Code – contagem e proporcao de cada classe
classes = pd.DataFrame({
    "qtd": df[COLUNA_CLASSE].value_counts().reindex(ORDEM_CLASSES),
    "pct": (df[COLUNA_CLASSE].value_counts(normalize=True) * 100).reindex(ORDEM_CLASSES).round(2),
})
classes

In [ ]:
# [IA] Claude Code – conferencia: a classe bate com o IMC (peso / altura^2) calculado?
# Faixas da OMS usadas pelos autores do dataset.
df_imc = df.assign(IMC=df["Weight"] / df["Height"] ** 2)
df_imc.groupby(COLUNA_CLASSE)["IMC"].describe().reindex(ORDEM_CLASSES).round(1)

## Observações do pilar estrutural

_(texto do grupo)_

## 2. Pilar distribucional

### 2.1 Variáveis numéricas

In [ ]:
# [IA] Claude Code – medidas resumo + assimetria e curtose de cada variavel numerica
resumo = df[colunas_numericas].describe().T
resumo["assimetria"] = df[colunas_numericas].skew()
resumo["curtose"] = df[colunas_numericas].kurt()
resumo.round(3)

In [ ]:
# [IA] Claude Code – histogramas das variaveis numericas (com curva de densidade)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, coluna in zip(axes.flat, colunas_numericas):
    sns.histplot(df[coluna], bins=30, kde=True, ax=ax, color="#4C72B0")
    ax.axvline(df[coluna].mean(), color="#C44E52", linestyle="--", linewidth=1.5, label="media")
    ax.axvline(df[coluna].median(), color="#55A868", linestyle="-", linewidth=1.5, label="mediana")
    ax.set_title(dicionario.loc[coluna, "significado"])
    ax.set_xlabel(coluna)
    ax.set_ylabel("Quantidade de pessoas")
    ax.legend(fontsize=8)
fig.suptitle("Obesity: distribuicao das variaveis numericas", fontsize=15)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_histogramas_numericas.png", dpi=150, bbox_inches="tight")
plt.show()

### Leitura das distribuições numéricas

_(texto do grupo)_

### 2.2 Variáveis categóricas

In [ ]:
# [IA] Claude Code – frequencias absolutas e relativas de cada variavel categorica
for coluna in colunas_categoricas.drop(COLUNA_CLASSE):
    tabela = pd.DataFrame({
        "qtd": df[coluna].value_counts(),
        "pct": (df[coluna].value_counts(normalize=True) * 100).round(2),
    })
    print(f"\n{coluna} ({dicionario.loc[coluna, 'significado']})")
    print(tabela.to_string())

In [ ]:
# [IA] Claude Code – graficos de barras das variaveis categoricas (barras horizontais ordenadas)
categoricas_sem_classe = colunas_categoricas.drop(COLUNA_CLASSE)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, coluna in zip(axes.flat, categoricas_sem_classe):
    contagem = df[coluna].value_counts().sort_values()
    ax.barh(contagem.index.astype(str), contagem.values, color="#4C72B0")
    for y, v in enumerate(contagem.values):
        ax.text(v, y, f" {v / len(df) * 100:.1f}%", va="center", fontsize=8)
    ax.set_title(dicionario.loc[coluna, "significado"])
    ax.set_xlabel("Quantidade de pessoas")
    ax.set_ylabel(coluna)
    ax.set_xlim(0, contagem.max() * 1.2)
fig.suptitle("Obesity: distribuicao das variaveis categoricas", fontsize=15)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_barras_categoricas.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# [IA] Claude Code – distribuicao da variavel de classe (na ordem do nivel de peso)
fig, ax = plt.subplots(figsize=(10, 5))
contagem = df[COLUNA_CLASSE].value_counts().reindex(ORDEM_CLASSES)
ax.bar(contagem.index, contagem.values, color="#4C72B0")
for x, v in enumerate(contagem.values):
    ax.text(x, v, f"{v}", ha="center", va="bottom", fontsize=9)
ax.set_title("Obesity: quantidade de pessoas em cada classe (NObeyesdad)")
ax.set_xlabel("Classe (nivel de peso)")
ax.set_ylabel("Quantidade de pessoas")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_distribuicao_classes.png", dpi=150, bbox_inches="tight")
plt.show()

### Leitura das distribuições categóricas e da classe

_(texto do grupo)_

## 3. Outliers

Dois critérios: **IQR** (abaixo de Q1 − 1,5·IQR ou acima de Q3 + 1,5·IQR) e **z-score** (|z| > 3).
As escalas de questionário (FCVC, NCP, CH2O, FAF, TUE) têm faixa fechada, então um valor "extremo"
nelas é uma resposta válida da escala, não um erro.

In [ ]:
# [IA] Claude Code – contagem de outliers por variavel pelos dois criterios
def limites_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

linhas_out = []
for coluna in colunas_numericas:
    inf, sup = limites_iqr(df[coluna])
    z = (df[coluna] - df[coluna].mean()) / df[coluna].std()
    linhas_out.append({
        "variavel": coluna,
        "limite_inferior_iqr": round(inf, 2),
        "limite_superior_iqr": round(sup, 2),
        "qtd_outliers_iqr": int(((df[coluna] < inf) | (df[coluna] > sup)).sum()),
        "qtd_outliers_z3": int((z.abs() > 3).sum()),
    })
outliers = pd.DataFrame(linhas_out).set_index("variavel")
outliers["pct_outliers_iqr"] = (outliers["qtd_outliers_iqr"] / len(df) * 100).round(2)
outliers

In [ ]:
# [IA] Claude Code – boxplots das variaveis numericas (cada uma na sua escala)
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, coluna in zip(axes.flat, colunas_numericas):
    sns.boxplot(y=df[coluna], ax=ax, color="#8DA0CB", flierprops={"marker": "o", "markersize": 3})
    ax.set_title(f"{coluna}: {outliers.loc[coluna, 'qtd_outliers_iqr']} outliers (IQR)")
    ax.set_xlabel("")
    ax.set_ylabel(coluna)
fig.suptitle("Obesity: boxplots das variaveis numericas", fontsize=15)
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_boxplots_numericas.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# [IA] Claude Code – boxplots de Age, Height e Weight separados por classe
# Um valor que e outlier no geral pode ser normal dentro da propria classe (ex.: peso alto em Obesity_Type_III).
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, coluna in zip(axes, ["Age", "Height", "Weight"]):
    sns.boxplot(data=df, x=COLUNA_CLASSE, y=coluna, order=ORDEM_CLASSES, ax=ax, color="#8DA0CB",
                flierprops={"marker": "o", "markersize": 3})
    ax.set_title(f"{coluna} por classe")
    ax.set_xlabel("Classe (nivel de peso)")
    ax.set_ylabel(coluna)
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right")
fig.tight_layout()
fig.savefig(PASTA_SAIDA / "eda_boxplots_por_classe.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# [IA] Claude Code – quem sao os outliers de Age, Height e Weight (para avaliar se sao plausiveis)
for coluna in ["Age", "Height", "Weight"]:
    inf, sup = limites_iqr(df[coluna])
    fora = df[(df[coluna] < inf) | (df[coluna] > sup)]
    print(f"\n{coluna}: {len(fora)} outliers | faixa dos outliers: {fora[coluna].min():.2f} a {fora[coluna].max():.2f}")
    print(fora[COLUNA_CLASSE].value_counts().to_string())

### Discussão dos outliers

_(texto do grupo: os valores são plausíveis? são erro de digitação ou pessoas reais?)_

In [ ]:
# [IA] Claude Code – aplicar o tratamento escolhido pelo grupo
# Opcoes:
#   "manter":      nenhum valor e alterado (outliers considerados valores reais)
#   "winsorizar":  valores fora dos limites do IQR sao trocados pelo proprio limite
#   "remover":     linhas com outlier (IQR) em VARIAVEIS_TRATADAS sao removidas
TRATAMENTO = "manter"  # (decisao do grupo)
VARIAVEIS_TRATADAS = ["Age", "Height", "Weight", "NCP"]  # (decisao do grupo)

df_tratado = df.copy()
if TRATAMENTO == "manter":
    pass
elif TRATAMENTO == "winsorizar":
    for coluna in VARIAVEIS_TRATADAS:
        inf, sup = limites_iqr(df[coluna])
        df_tratado[coluna] = df_tratado[coluna].clip(inf, sup)
elif TRATAMENTO == "remover":
    mascara = pd.Series(False, index=df.index)
    for coluna in VARIAVEIS_TRATADAS:
        inf, sup = limites_iqr(df[coluna])
        mascara |= (df[coluna] < inf) | (df[coluna] > sup)
    df_tratado = df_tratado[~mascara]
else:
    raise ValueError(f"Tratamento desconhecido: {TRATAMENTO}")

print(f"Tratamento: {TRATAMENTO} | linhas antes: {len(df)} | linhas depois: {len(df_tratado)}")

### Justificativa do tratamento

_(texto do grupo)_

In [ ]:
# [IA] Claude Code – salvar o dataset apos o tratamento de outliers (usado nas proximas etapas)
df_tratado.to_csv(PASTA_SAIDA / "obesity_tratado.csv", index=False)
print("Salvo em", PASTA_SAIDA / "obesity_tratado.csv")